# metafopy showcase

[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/bnomis/metafopy/HEAD?labpath=docs%2Fnotebooks%2Fshowcase.ipynb)

This notebook shows what meta-analysis looks like when it lives in Python and Jupyter. Every number comes from metafopy; the charts are
[Altair](https://altair-viz.github.io/) (Vega-Lite) charts that stay interactive on the static docs site, because the fits are computed here in
Python and the browser only filters them. The statistics are those of the R package [metafor](https://www.metafor-project.org/)
(Viechtbauer, 2010, [doi:10.18637/jss.v036.i03](https://doi.org/10.18637/jss.v036.i03)); the plots are free to look different, see
[Plots compared with metafor](../differences-plots.md) for the matplotlib versions that mirror R.

The story: a main thread on the BCG vaccine trials (Colditz et al., 1994, [doi:10.1001/jama.1994.03520050036011](https://doi.org/10.1001/jama.1994.03520050036011)),
then two side trips: a multilevel model for schools nested in districts (Konstantopoulos, 2011,
[doi:10.1002/jrsm.35](https://doi.org/10.1002/jrsm.35)) and publication bias in passive smoking and lung cancer (Hackshaw et al., 1997,
[doi:10.1136/bmj.315.7114.980](https://doi.org/10.1136/bmj.315.7114.980)), and a generated write-up with exported figures.

Hover for tooltips, click a study to highlight it everywhere, drag in the funnel plot to brush, and use the controls above the forest plot.

In [ ]:
import pathlib
import tempfile

import altair
import IPython.display
import numpy
import pandas

import metafopy
from metafopy import cumul, datasets, escalc, fsn, gosh, influence, predict, ranef, regtest, rma, rma_mv, selmodel, trimfill

GRAPHITE = '#3D3F44'
BLUE = '#268BD2'
GREEN = '#2AA198'
YELLOW = '#B58900'
RED = '#DC322F'
MUTED = '#93A1A1'

altair.renderers.enable('html', embed_options={'actions': False})
altair.data_transformers.disable_max_rows()
print(f'metafopy {metafopy.__version__}, altair {altair.__version__}')

## 1. The data and a first model

`escalc` turns the 2x2 tables of the 13 trials into log risk ratios and sampling variances; `rma` fits the random-effects model.
Results render as tables here (the printed text of R's `summary()` is still what `print(fit)` gives).

In [ ]:
bcg = datasets.load('dat.bcg')
es = escalc('RR', ai='tpos', bi='tneg', ci='cpos', di='cneg', data=bcg)
es['study'] = bcg['author'] + ', ' + bcg['year'].astype(str)
es[['study', 'year', 'ablat', 'alloc', 'yi', 'vi']].head()

In [ ]:
fit = rma(yi='yi', vi='vi', data=es)
fit

The pooled estimate is on the log scale. A prediction interval says where the effect of a *new* trial would fall, which is much wider
than the confidence interval when the trials disagree as much as these do.

In [ ]:
pred = predict(fit, transf=numpy.exp)
print(
    f'Pooled risk ratio {float(pred.pred[0]):.2f}, 95% CI [{float(pred.ci_lb[0]):.2f}, {float(pred.ci_ub[0]):.2f}], '
    f'95% prediction interval [{float(pred.pi_lb[0]):.2f}, {float(pred.pi_ub[0]):.2f}]'
)

## 2. Everything a study contributes, in one frame

The charts below share one tidy frame with a row per study: effect and interval, weight, the coordinates of the funnel and
[Baujat](https://doi.org/10.1002/sim.1186) plots, and the influence diagnostics of `influence()`.

In [ ]:
yi = numpy.asarray(es['yi'], dtype=float)
vi = numpy.asarray(es['vi'], dtype=float)
inf = influence(fit).to_frame()

fixed = rma(yi='yi', vi='vi', data=es, method='EE')
theta_fixed = float(fixed.beta[0])
baujat_y = []
for i in range(len(yi)):
    keep = numpy.arange(len(yi)) != i
    loo = rma(yi=yi[keep], vi=vi[keep], method='EE')
    baujat_y.append((theta_fixed - float(loo.beta[0])) ** 2 / float(loo.se[0]) ** 2)

weight = 1 / (vi + fit.tau2)
studies = pandas.DataFrame(
    {
        'study': es['study'],
        'alloc': bcg['alloc'],
        'ablat': bcg['ablat'],
        'n': bcg[['tpos', 'tneg', 'cpos', 'cneg']].sum(axis=1),
        'rr': numpy.exp(yi),
        'lo': numpy.exp(yi - 1.96 * numpy.sqrt(vi)),
        'hi': numpy.exp(yi + 1.96 * numpy.sqrt(vi)),
        'yi': yi,
        'sei': numpy.sqrt(vi),
        'weight': 100 * weight / weight.sum(),
        'baujat_x': (yi - theta_fixed) ** 2 / vi,
        'baujat_y': baujat_y,
        'cook_d': numpy.asarray(inf['cook.d'], dtype=float),
        'rstudent': numpy.asarray(inf['rstudent'], dtype=float),
    }
)
order = list(studies.sort_values('ablat')['study'])
studies.round(3).head()

## 3. Fit every estimator, with and without each trial

The pooled estimate depends on two choices: which estimator of the between-trial variance `tau^2` you use (metafopy has seven, see
[Veroniki et al. (2016)](https://doi.org/10.1002/jrsm.1164)), and whether you leave one trial out to see if it drives the result.
Here every combination is fitted once, in Python: 7 estimators x 14 options (leave out none, or one of the 13 trials) = 98 fits.
Each row of the `pooled` frame is one fit, with its pooled risk ratio, intervals, `tau^2` and I^2. The next cell prints the range
of the pooled risk ratio (`est`) and of `tau^2` over the 14 options for each estimator; section 4 turns the same frame into
controls.

In [ ]:
estimators = ['DL', 'EB', 'HE', 'ML', 'PM', 'REML', 'SJ']
rows = []
for method in estimators:
    for drop in ['(none)', *studies['study']]:
        keep = (studies['study'] != drop).to_numpy()
        f = rma(yi=yi[keep], vi=vi[keep], method=method)
        p = predict(f)
        rows.append(
            {
                'method': method,
                'drop': drop,
                'est': float(numpy.exp(f.beta[0])),
                'lo': float(numpy.exp(f.ci_lb[0])),
                'hi': float(numpy.exp(f.ci_ub[0])),
                'pi_lo': float(numpy.exp(numpy.ravel(p.pi_lb)[0])),
                'pi_hi': float(numpy.exp(numpy.ravel(p.pi_ub)[0])),
                'tau2': float(f.tau2),
                'I2': float(f.I2),
            }
        )
pooled = pandas.DataFrame(rows)
pooled.groupby('method')[['est', 'tau2']].agg(['min', 'max']).round(3)

## 4. Linked forest, funnel, Baujat and meta-regression plots

Click a study (shift-click for several) or brush a region of the funnel plot: the same studies light up in all four plots.
The radio buttons and the "Leave out" drop-down below the charts pick a row of the `pooled` frame of section 3, the `tau^2` estimator and the omitted trial; the red line and band in the forest plot,
and the numbers above it, follow. The bubble plot is the meta-regression on absolute latitude, the moderator that explains most of the
heterogeneity (`predict(..., newmods=)` draws the band).

In [ ]:
mr = rma(yi='yi', vi='vi', mods='~ ablat', data=es)
grid = numpy.linspace(10, 60, 51)
pr = predict(mr, newmods=grid)
band = pandas.DataFrame(
    {'ablat': grid, 'pred': numpy.exp(numpy.ravel(pr.pred)), 'lo': numpy.exp(numpy.ravel(pr.ci_lb)), 'hi': numpy.exp(numpy.ravel(pr.ci_ub))}
)

pick = altair.selection_point(fields=['study'], on='click', clear='dblclick', toggle='true')
brush = altair.selection_interval(encodings=['x', 'y'], name='brush')
method = altair.param(name='method', value='REML', bind=altair.binding_radio(options=estimators, name='tau² estimator  '))
drop = altair.param(name='drop', value='(none)', bind=altair.binding_select(options=['(none)', *order], name='Leave out  '))
highlight = pick | brush
dim = altair.condition(highlight, altair.value(1), altair.value(0.2))

rr_scale = altair.Scale(type='log', domain=[0.05, 8], clamp=True)
rr_axis = altair.Axis(values=[0.05, 0.1, 0.25, 0.5, 1, 2, 4, 8], format='~g')
tooltip = [
    altair.Tooltip('study:N', title='Study'),
    altair.Tooltip('rr:Q', title='Risk ratio', format='.2f'),
    altair.Tooltip('lo:Q', title='95% CI low', format='.2f'),
    altair.Tooltip('hi:Q', title='95% CI high', format='.2f'),
    altair.Tooltip('weight:Q', title='Weight %', format='.1f'),
    altair.Tooltip('ablat:Q', title='Abs. latitude'),
    altair.Tooltip('alloc:N', title='Allocation'),
    altair.Tooltip('cook_d:Q', title="Cook's distance", format='.3f'),
]
alloc_color = altair.Color('alloc:N', title='Allocation', scale=altair.Scale(domain=['random', 'alternate', 'systematic'], range=[BLUE, GREEN, YELLOW]))

base = altair.Chart(studies)
y_study = altair.Y('study:N', sort=order, title=None, axis=altair.Axis(labelLimit=200))
forest_ci = base.mark_rule(strokeWidth=2).encode(
    x=altair.X('lo:Q', scale=rr_scale, title='Risk ratio (log scale)', axis=rr_axis), x2='hi:Q', y=y_study, color=alloc_color, opacity=dim
)
forest_pt = (
    base.mark_square()
    .encode(
        x=altair.X('rr:Q', scale=rr_scale),
        y=y_study,
        size=altair.Size('weight:Q', scale=altair.Scale(range=[30, 300]), legend=None),
        color=alloc_color,
        opacity=dim,
        tooltip=tooltip,
    )
    .add_params(pick)
)
null_line = altair.Chart(pandas.DataFrame({'x': [1]})).mark_rule(color=MUTED, strokeDash=[4, 3]).encode(x='x:Q')
pooled_sel = altair.Chart(pooled).transform_filter((altair.datum.method == method) & (altair.datum.drop == drop))
pooled_line = pooled_sel.mark_rule(color=RED, strokeWidth=1.5).encode(x=altair.X('est:Q', scale=rr_scale))
pooled_band = pooled_sel.mark_rect(color=RED, opacity=0.08).encode(x=altair.X('lo:Q', scale=rr_scale), x2='hi:Q')
forest = (
    (pooled_band + pooled_line + null_line + forest_ci + forest_pt)
    .properties(width=420, height=300, title='Forest plot (sorted by latitude)')
    .add_params(method, drop)
)
pooled_text = (
    pooled_sel.transform_calculate(
        txt='"RR " + format(datum.est, ".2f") + " [" + format(datum.lo, ".2f") + ", " + format(datum.hi, ".2f") + "]   PI ["'
        ' + format(datum.pi_lo, ".2f") + ", " + format(datum.pi_hi, ".2f") + "]   τ² " + format(datum.tau2, ".3f") + "   I² " + format(datum.I2, ".1f") + "%"'
    )
    .mark_text(align='left', fontSize=13, color=GRAPHITE, fontWeight='bold')
    .encode(text='txt:N')
    .properties(width=420, height=18)
)

se_max = float(studies['sei'].max()) * 1.1
funnel_tri = pandas.DataFrame({'se': numpy.linspace(0, se_max, 30)})
funnel_tri['lo'] = numpy.exp(float(fit.beta[0]) - 1.96 * funnel_tri['se'])
funnel_tri['hi'] = numpy.exp(float(fit.beta[0]) + 1.96 * funnel_tri['se'])
se_scale = altair.Scale(reverse=True, domain=[0, se_max])
funnel_chart = (
    altair.Chart(funnel_tri)
    .mark_area(color=MUTED, opacity=0.15)
    .encode(
        x=altair.X('lo:Q', scale=rr_scale, title='Risk ratio (log scale)', axis=rr_axis), x2='hi:Q', y=altair.Y('se:Q', scale=se_scale, title='Standard error')
    )
    + base.mark_circle(size=90, stroke='white', strokeWidth=1)
    .encode(x=altair.X('rr:Q', scale=rr_scale), y=altair.Y('sei:Q', scale=se_scale), color=alloc_color, opacity=dim, tooltip=tooltip)
    .add_params(brush)
).properties(width=300, height=260, title='Funnel plot (drag to brush)')

baujat_chart = (
    base.mark_circle(size=90, stroke='white', strokeWidth=1)
    .encode(
        x=altair.X('baujat_x:Q', title='Contribution to Q'),
        y=altair.Y('baujat_y:Q', title='Influence on pooled estimate'),
        color=alloc_color,
        opacity=dim,
        tooltip=tooltip,
    )
    .properties(width=300, height=260, title='Baujat plot')
)

lat_axis = altair.Axis(values=[0.1, 0.25, 0.5, 1, 2], format='~g')
bubble_chart = (
    altair.Chart(band)
    .mark_area(color=BLUE, opacity=0.12)
    .encode(
        x=altair.X('ablat:Q', title='Absolute latitude (°)'),
        y=altair.Y('lo:Q', scale=altair.Scale(type='log', domain=[0.1, 2]), title='Risk ratio', axis=lat_axis),
        y2='hi:Q',
    )
    + altair.Chart(band).mark_line(color=BLUE, strokeWidth=2).encode(x='ablat:Q', y='pred:Q')
    + base.mark_circle(stroke='white', strokeWidth=1).encode(
        x='ablat:Q',
        y=altair.Y('rr:Q', scale=altair.Scale(type='log')),
        size=altair.Size('weight:Q', scale=altair.Scale(range=[30, 500]), legend=None),
        color=alloc_color,
        opacity=dim,
        tooltip=tooltip,
    )
).properties(width=420, height=260, title='Meta-regression on latitude (REML)')

dashboard = (
    altair.vconcat(altair.hconcat(altair.vconcat(pooled_text, forest), funnel_chart), altair.hconcat(bubble_chart, baujat_chart))
    .resolve_scale(size='independent')
    .configure(font='Source Sans 3, Helvetica, Arial, sans-serif')
    .configure_view(stroke=None)
    .configure_axis(labelColor=GRAPHITE, titleColor=GRAPHITE, gridColor='#EEE8D5', domainColor=MUTED, tickColor=MUTED)
    .configure_title(color=GRAPHITE, fontSize=13, anchor='start', fontWeight=600)
    .configure_legend(labelColor=GRAPHITE, titleColor=GRAPHITE, orient='bottom')
)
dashboard

Things to try: leave out *Hart & Sutherland, 1977* and watch the pooled estimate and `tau^2` change; pick `DL` and then `SJ`; brush the
small, imprecise trials at the bottom of the funnel plot and see where they sit in the Baujat plot.

## 5. Cumulative meta-analysis

`cumul()` adds the trials one at a time in the order you give, here by publication year. The chart lets you change the estimator
and shows how the estimate settles as the evidence accumulates.

In [ ]:
years = bcg['year'].to_numpy()
cum_rows = []
for m in ['DL', 'HE', 'PM', 'REML']:
    c = cumul(rma(yi='yi', vi='vi', data=es, method=m, slab=es['study']), order=years)
    cum_rows.append(
        pandas.DataFrame(
            {
                'method': m,
                'step': numpy.arange(1, len(c.slab) + 1),
                'study': list(c.slab),
                'year': numpy.sort(years),
                'est': numpy.exp(numpy.asarray(c.estimate)),
                'lo': numpy.exp(numpy.asarray(c.ci_lb)),
                'hi': numpy.exp(numpy.asarray(c.ci_ub)),
            }
        )
    )
cumulative = pandas.concat(cum_rows, ignore_index=True)

cum_method = altair.param(name='cum_method', value='REML', bind=altair.binding_radio(options=['DL', 'HE', 'PM', 'REML'], name='tau² estimator  '))
cum_base = altair.Chart(cumulative).transform_filter(altair.datum.method == cum_method)
cum_x = altair.X('step:O', title='Trials added (in order of publication year)', axis=altair.Axis(labelAngle=0))
cum_chart = (
    (
        cum_base.mark_area(color=BLUE, opacity=0.15).encode(
            x=cum_x,
            y=altair.Y(
                'lo:Q', scale=altair.Scale(type='log', domain=[0.1, 3]), title='Pooled risk ratio', axis=altair.Axis(values=[0.1, 0.25, 0.5, 1, 2], format='~g')
            ),
            y2='hi:Q',
        )
        + cum_base.mark_line(color=BLUE, point=altair.OverlayMarkDef(filled=True, size=60)).encode(
            x=cum_x,
            y='est:Q',
            tooltip=['study:N', 'year:Q', altair.Tooltip('est:Q', format='.2f'), altair.Tooltip('lo:Q', format='.2f'), altair.Tooltip('hi:Q', format='.2f')],
        )
        + altair.Chart(pandas.DataFrame({'y': [1]})).mark_rule(color=MUTED, strokeDash=[4, 3]).encode(y='y:Q')
    )
    .properties(width=620, height=260, title='Cumulative meta-analysis')
    .add_params(cum_method)
)
cum_chart

## 6. GOSH: all subsets of the studies

[Olkin, Dahabreh and Trikalinos (2012)](https://doi.org/10.1002/jrsm.1041) fit the model to every possible subset of studies (here
2^13 - 1 = 8191 fits) and plot the pooled estimate against `I^2`. Clusters point at a subgroup of studies that behave differently. Brush a
region of the scatterplot: the histogram shows the estimates in the brush, and the bars show how often each trial is in those subsets.

In [ ]:
g = gosh(fit)
gosh_frame = g.res[['estimate', 'I2']].copy()
gosh_frame['estimate'] = numpy.exp(gosh_frame['estimate']).round(4)
gosh_frame['I2'] = gosh_frame['I2'].round(2)
incl = numpy.asarray(g.incl, dtype=int)
labels = list(studies['study'])
for j, label in enumerate(labels):
    gosh_frame[label] = incl[:, j]

gosh_brush = altair.selection_interval(name='gosh_brush')
scatter = (
    altair.Chart(gosh_frame)
    .mark_circle(size=8, opacity=0.25)
    .encode(
        x=altair.X('estimate:Q', scale=altair.Scale(type='log'), title='Pooled risk ratio', axis=altair.Axis(values=[0.2, 0.3, 0.5, 0.75, 1], format='~g')),
        y=altair.Y('I2:Q', title='I² (%)'),
        color=altair.condition(gosh_brush, altair.value(BLUE), altair.value(MUTED)),
    )
    .add_params(gosh_brush)
    .properties(width=360, height=300, title='All 8191 subsets (drag to brush)')
)
hist = (
    altair.Chart(gosh_frame)
    .transform_filter(gosh_brush)
    .mark_bar(color=BLUE)
    .encode(
        x=altair.X('estimate:Q', bin=altair.Bin(maxbins=30), scale=altair.Scale(type='log'), title='Pooled risk ratio in brush'),
        y=altair.Y('count()', title='Subsets'),
    )
    .properties(width=300, height=120)
)
inclusion = (
    altair.Chart(gosh_frame)
    .transform_filter(gosh_brush)
    .transform_fold(labels, as_=['study', 'in'])
    .mark_bar(color=GREEN)
    .encode(
        y=altair.Y('study:N', sort=order, title=None),
        x=altair.X('mean(in):Q', scale=altair.Scale(domain=[0, 1]), title='Share of brushed subsets containing the trial'),
    )
    .properties(width=300, height=170)
)
(scatter | (hist & inclusion)).configure_axis(labelColor=GRAPHITE, titleColor=GRAPHITE, gridColor='#EEE8D5').configure_title(color=GRAPHITE, anchor='start')

## Side trip: schools nested in districts

`dat.konstantopoulos2011` has 56 studies of the effect of modified school calendars on achievement, from 11 districts. Studies in one
district are not independent, so `rma_mv` adds a random effect for district and one for school within district (Konstantopoulos, 2011).

In [ ]:
kdat = datasets.load('dat.konstantopoulos2011')
mv = rma_mv(yi='yi', V='vi', random='~ 1 | district/school', data=kdat)
mv

The two variance components are about equal in size to the typical sampling variance. `ranef` predicts the effect of each district with
its prediction interval; the chart shows these over the observed study effects.

In [ ]:
districts = ranef(mv)['district'].reset_index(names='district')
districts['district'] = districts['district'].astype(str)
districts['est'] = districts['intrcpt'] + float(mv.beta[0])
districts['lo'] = districts['pi.lb'] + float(mv.beta[0])
districts['hi'] = districts['pi.ub'] + float(mv.beta[0])
kstudies = kdat.assign(district=kdat['district'].astype(str), se=numpy.sqrt(kdat['vi']))
dist_order = list(districts.sort_values('est')['district'])

district_pick = altair.selection_point(fields=['district'], on='click', clear='dblclick')
dist_y = altair.Y('district:N', sort=dist_order, title='District')
dist_bars = (
    altair.Chart(districts)
    .mark_rule(strokeWidth=3, color=BLUE)
    .encode(
        x=altair.X('lo:Q', title='Standardised mean difference'),
        x2='hi:Q',
        y=dist_y,
        opacity=altair.condition(district_pick, altair.value(1), altair.value(0.3)),
    )
)
dist_pts = (
    altair.Chart(districts)
    .mark_point(filled=True, size=90, color=GRAPHITE)
    .encode(x='est:Q', y=dist_y, tooltip=['district:N', altair.Tooltip('est:Q', title='District effect', format='.2f')])
    .add_params(district_pick)
)
study_pts = (
    altair.Chart(kstudies)
    .mark_circle(size=40, color=YELLOW)
    .encode(
        x='yi:Q',
        y=dist_y,
        opacity=altair.condition(district_pick, altair.value(0.9), altair.value(0.2)),
        tooltip=['district:N', 'school:Q', 'year:Q', 'yi:Q', altair.Tooltip('se:Q', format='.2f')],
    )
)
overall = altair.Chart(pandas.DataFrame({'x': [float(mv.beta[0])]})).mark_rule(color=RED, strokeDash=[4, 3]).encode(x='x:Q')
(overall + study_pts + dist_bars + dist_pts).properties(
    width=520, height=300, title='District effects (black, with 95% prediction interval) and the school effects within them (yellow); red = overall'
).configure_title(color=GRAPHITE, fontSize=12, anchor='start').configure_axis(labelColor=GRAPHITE, titleColor=GRAPHITE, gridColor='#EEE8D5')

## Side trip: publication bias

`dat.hackshaw1998` pools 37 studies of lung cancer risk and passive smoking. The funnel plot below marks the studies that
the trim-and-fill method (Duval and Tweedie, 2000, [doi:10.1111/j.0006-341X.2000.00455.x](https://doi.org/10.1111/j.0006-341X.2000.00455.x))
would add to make it symmetric; hover them to see how much each would change the picture.

In [ ]:
hack = datasets.load('dat.hackshaw1998')
fit_h = rma(yi='yi', vi='vi', data=hack)
tf = trimfill(fit_h)
tf_frame = pandas.DataFrame({'yi': numpy.asarray(tf.yi), 'sei': numpy.sqrt(numpy.asarray(tf.vi)), 'imputed': numpy.asarray(tf.fill)})
tf_frame['kind'] = numpy.where(tf_frame['imputed'], 'filled in', 'observed')
se_top = float(tf_frame['sei'].max()) * 1.05
cone = pandas.DataFrame({'se': numpy.linspace(0, se_top, 30)})
cone['lo'] = float(fit_h.beta[0]) - 1.96 * cone['se']
cone['hi'] = float(fit_h.beta[0]) + 1.96 * cone['se']
se_rev = altair.Scale(reverse=True, domain=[0, se_top])
funnel_h = (
    (
        altair.Chart(cone)
        .mark_area(color=MUTED, opacity=0.15)
        .encode(x=altair.X('lo:Q', title='Log odds ratio'), x2='hi:Q', y=altair.Y('se:Q', scale=se_rev, title='Standard error'))
        + altair.Chart(pandas.DataFrame({'x': [float(fit_h.beta[0])]})).mark_rule(color=MUTED).encode(x='x:Q')
        + altair.Chart(tf_frame)
        .mark_point(filled=True, size=70, strokeWidth=1.5)
        .encode(
            x='yi:Q',
            y=altair.Y('sei:Q', scale=se_rev),
            stroke=altair.Color('kind:N', title=None, scale=altair.Scale(domain=['observed', 'filled in'], range=[BLUE, RED])),
            fill=altair.condition(altair.datum.imputed, altair.value('white'), altair.value(BLUE)),
            tooltip=['kind:N', altair.Tooltip('yi:Q', title='Log odds ratio', format='.2f'), altair.Tooltip('sei:Q', title='SE', format='.2f')],
        )
    )
    .properties(width=460, height=320, title=f'Trim and fill: {int(tf_frame["imputed"].sum())} studies added on the left')
    .configure_title(color=GRAPHITE, anchor='start')
)
funnel_h

The usual tests and corrections side by side. The regression test (Egger et al., 1997, [doi:10.1136/bmj.315.7109.629](https://doi.org/10.1136/bmj.315.7109.629)),
the fail-safe N (Rosenthal, 1979) and a half-normal selection model (Vevea and Hedges, 1995; Preston et al., 2004) each look at the
asymmetry in a different way.

In [ ]:
summary_rows = pandas.DataFrame(
    [
        {'analysis': 'Random-effects model', 'estimate': float(fit_h.beta[0]), 'ci.lb': float(fit_h.ci_lb[0]), 'ci.ub': float(fit_h.ci_ub[0])},
        {'analysis': 'Trim and fill', 'estimate': float(tf.beta[0]), 'ci.lb': float(tf.ci_lb[0]), 'ci.ub': float(tf.ci_ub[0])},
        {
            'analysis': 'Selection model (half-normal)',
            **dict(
                zip(
                    ['estimate', 'ci.lb', 'ci.ub'],
                    [float(s) for s in (lambda sm: (sm.beta[0], sm.ci_lb[0], sm.ci_ub[0]))(selmodel(fit_h, type='halfnorm'))],
                    strict=True,
                )
            ),
        },
    ]
)
summary_rows[['estimate', 'ci.lb', 'ci.ub']] = numpy.exp(summary_rows[['estimate', 'ci.lb', 'ci.ub']])
summary_rows.columns = ['analysis', 'odds ratio', 'ci.lb', 'ci.ub']
summary_rows.round(3)

In [ ]:
regtest(fit_h)

In [ ]:
fsn(fit_h)

## A generated write-up

metafor's `reporter()` writes a report of a fit; metafopy does not port it, because a notebook does the job better: the text
below is built from the fits above, so it cannot disagree with them.

In [ ]:
p = predict(fit, transf=numpy.exp)
q = mr.QM
qe_p = '< .0001' if fit.QEp < 1e-4 else f'= {fit.QEp:.4f}'
report = f"""
### Results

We pooled {fit.k} randomised trials of BCG vaccination with a random-effects model ({fit.method} estimate of τ²). The pooled risk ratio
of tuberculosis was {float(p.pred[0]):.2f} (95% CI {float(p.ci_lb[0]):.2f} to {float(p.ci_ub[0]):.2f}; 95% prediction interval
{float(p.pi_lb[0]):.2f} to {float(p.pi_ub[0]):.2f}). The trials were heterogeneous (Q({fit.k - 1}) = {fit.QE:.1f}, p {qe_p};
τ² = {fit.tau2:.3f}; I² = {fit.I2:.1f}%).

Absolute latitude explained a large part of the heterogeneity: each additional degree of latitude changed the log risk ratio by
{float(mr.beta[1]):.3f} (95% CI {float(mr.ci_lb[1]):.3f} to {float(mr.ci_ub[1]):.3f}; QM(1) = {q:.1f}), with R² = {mr.R2:.1f}%.

Across the {len(pooled['method'].unique())} τ² estimators the pooled risk ratio ranged from
{pooled.query("drop == '(none)'")['est'].min():.3f} to {pooled.query("drop == '(none)'")['est'].max():.3f}.
"""
IPython.display.Markdown(report)

### Exported figures

The same fits drive static figures for a paper: the matplotlib forest plot of metafopy, and the Altair charts saved as SVG or PNG
with [vl-convert](https://github.com/vega/vl-convert) (no browser needed). The files go to a temporary directory; point `out` anywhere you like.

In [ ]:
import matplotlib
import matplotlib.pyplot

from metafopy import forest

out = pathlib.Path(tempfile.mkdtemp(prefix='metafopy-showcase-'))
fit_labelled = rma(yi='yi', vi='vi', data=es, slab=es['study'])
matplotlib.pyplot.figure(figsize=(8, 6))
forest(fit_labelled, atransf=numpy.exp, header=True)
matplotlib.pyplot.savefig(out / 'forest.png', dpi=150, bbox_inches='tight')
matplotlib.pyplot.close()

dashboard.save(out / 'dashboard.svg')
dashboard.save(out / 'dashboard.png', scale_factor=2)
funnel_h.save(out / 'funnel-hackshaw.svg')
print(*sorted(f'{f.name} ({f.stat().st_size / 1024:.0f} KB)' for f in out.iterdir()), sep='\n')
IPython.display.Image(out / 'forest.png', width=520)

## Where next

* [User guide](../guide/models.md) and [function reference](../reference/index.md) for everything not shown here.
* [Coming from R](../from-r.md) if you know metafor already.
* Run this notebook yourself: `pip install "metafopy[interactive]" jupyterlab`, or use the Binder badge at the top.